# HoloMine Task 2: Prediksi Harga Properti dari Deskripsi Penjualan

Notebook ini isinya seluruh pengerjaan tim kami untuk task 2, mulai dari eksplorasi data
sampai file submission jadi.

Soalnya cukup tidak biasa. Biasanya prediksi harga rumah datang dengan kolom rapi berisi
luas tanah, jumlah kamar, tahun bangun. Di sini tidak ada satupun dari itu. Yang diberikan
cuma satu kolom teks berisi deskripsi iklan properti yang ditulis agen, lalu kami diminta
menebak harganya.

Struktur notebook:

1. Memahami metrik dulu sebelum menyentuh model
2. Eksplorasi data dan analisis ekor harga
3. Rekayasa fitur dari teks
4. Pemeriksaan pergeseran distribusi train dan test
5. Strategi validasi
6. Model bertingkat: TF-IDF, embedding, lalu stacking
7. Evaluasi dan analisis error
8. Submission

Total runtime sekitar satu jam di GPU T4 x2.

## 1. Metrik dulu, model belakangan

Metriknya MAE pada harga skala mentah:

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Ini menentukan banyak hal, jadi kami bahas di depan.

MAE diminimalkan oleh **median bersyarat**, bukan rata-rata bersyarat. Kalau kita melatih
model dengan MSE, yang keluar adalah estimasi rata-rata, dan pada data yang ekornya berat
rata-rata jauh di atas median. Untuk MAE itu jelas merugikan.

Kedua, median itu invarian terhadap transformasi monoton. Artinya
$\text{median}(y|x) = \exp(\text{median}(\log y|x))$. Jadi kami bisa melatih semua model
pada `log(listPrice)` dengan loss L1, lalu meng-`exp()` hasilnya, dan yang didapat tetap
median bersyarat pada skala asli. Melatih di ruang log jauh lebih stabil karena harganya
membentang dari puluhan ribu sampai puluhan juta.

Konsekuensi ketiga yang baru kami sadari belakangan: karena MAE menghitung selisih absolut
pada skala mentah, satu rumah 10 juta yang meleset 50 persen menyumbang error yang sama
dengan 100 rumah 500 ribu yang meleset 10 persen. Jadi nasib skor kami ditentukan oleh
segelintir properti mahal. Ini kami kejar di bagian analisis ekor.

In [ ]:
import os, re, gc, time, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

SEED = 42
NFOLD = 5
np.random.seed(SEED)

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#1a1a19", "#52514e", "#e6e6e3"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.labelcolor": INK2,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

def rupiah_free(x, pos):
    if x >= 1e6: return f"{x/1e6:.0f} jt"
    if x >= 1e3: return f"{x/1e3:.0f} rb"
    return f"{x:.0f}"

FMT = mticker.FuncFormatter(rupiah_free)

def tidy(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    if title: ax.set_title(title, color=INK, pad=12, loc="left")
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(axis=grid_axis, alpha=0.7)
    ax.set_axisbelow(True)
    return ax

In [ ]:
def find_data():
    for c in [os.getcwd()] + sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")):
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError("train.csv tidak ketemu")

DATA = find_data()
train = pd.read_csv(os.path.join(DATA, "train.csv"))
test = pd.read_csv(os.path.join(DATA, "test.csv"))
sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))

print("folder data :", DATA)
print("train       :", train.shape)
print("test        :", test.shape)
print("sample sub  :", sample.shape)
print()
print(train.dtypes)
print()
print("missing di train:", train.isna().sum().to_dict())
print("missing di test :", test.isna().sum().to_dict())

In [ ]:
# lihat satu contoh utuh biar kebayang bentuk teksnya
row = train.iloc[0]
print("id        :", row.id)
print("listPrice :", f"{row.listPrice:,.0f}")
print()
print(row.text[:900], "...")

Yang langsung kelihatan dari contoh di atas: teksnya iklan properti sungguhan, bukan hasil
template. Gaya bahasanya promosi, informasi teknis diselipkan di tengah kalimat, dan nama
entitas tertentu disensor jadi `[Redacted Entity]`.

Kami sempat curiga datanya dibangkitkan mesin, karena kalau iya biasanya ada pola yang bisa
dibalik dan itu jalan pintas yang besar. Jadi kami cek dulu.

In [ ]:
allt = pd.concat([train.text, test.text], ignore_index=True)

print("teks unik            :", f"{allt.nunique()} dari {len(allt)}", f"({allt.nunique()/len(allt):.4f})")

kal = []
for t in train.text.head(4000):
    kal += [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 25]
vc = pd.Series(kal).value_counts()
print("kalimat unik         :", f"{len(vc)} dari {len(kal)}", f"({len(vc)/len(kal):.3f})")
pat_caps = r"\b[A-Z]{4,}\b"
pat_kontak = r"\d{3}-\d{4}|@|www\.|http"
rasio_caps = train.text.str.contains(pat_caps).mean()
rasio_kontak = train.text.str.contains(pat_kontak).mean()
print("ada ALL-CAPS         :", f"{rasio_caps:.3f}")
print("ada nomor telp / URL :", f"{rasio_kontak:.3f}")
print()
print("5 kalimat yang paling sering berulang:")
for s, c in vc.head(5).items():
    print(f"  {c:3d}x  {s[:75]}")

Teksnya unik 100 persen, kalimatnya unik 99 persen, dan 18 persen mengandung kata kapital
semua seperti "PRICED TO SELL". Tidak ada template, tidak ada generator yang bisa dibalik.
Kalimat yang berulang cuma basa basi agen seperti "Schedule your showing today!".

Kesimpulannya datanya asli, dan satu-satunya jalan adalah memodelkan makna teksnya.

## 2. Distribusi harga

Ini variabel targetnya. Kami plot di skala log karena di skala asli grafiknya cuma satu
batang di kiri dan ekor panjang yang tidak kelihatan.

In [ ]:
y = train.listPrice.values
ly = np.log(np.clip(y, 1, None))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(np.clip(y, 1, None)), bins=70, color=BLUE, edgecolor="white", linewidth=0.4)
for v, lab in [(np.median(y), "median"), (y.mean(), "rata-rata")]:
    ax.axvline(np.log10(v), color=ORANGE, linewidth=2, linestyle="--")
    ax.annotate(f"{lab}\n{v:,.0f}", xy=(np.log10(v), ax.get_ylim()[1]*0.82),
                xytext=(6, 0), textcoords="offset points", color=INK, fontsize=9)
ax.set_xticks(range(0, 9))
ax.set_xticklabels([f"$10^{i}$" for i in range(9)])
tidy(ax, "Sebaran listPrice (sumbu x skala log)", "harga", "jumlah listing")
plt.tight_layout(); plt.show()

q = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
print(pd.Series({f"p{int(v*100)}": train.listPrice.quantile(v) for v in q}).apply(lambda v: f"{v:,.0f}"))
print()
print(f"min       : {y.min():,.0f}")
print(f"max       : {y.max():,.0f}")
print(f"rata-rata : {y.mean():,.0f}")
print(f"median    : {np.median(y):,.0f}")
print(f"std       : {y.std():,.0f}")
print(f"rasio rata-rata/median : {y.mean()/np.median(y):.2f}")

Rata-ratanya 1,68 kali median. Sebarannya miring ke kanan dan ekornya tebal sekali, dari
harga 1 dolar sampai 80 juta dolar.

Ada 3 listing berharga 1 dolar. Setelah dibaca teksnya, ketiganya properti yang sedang
dilelang online, jadi harga listingnya cuma placeholder. Kami biarkan saja, jumlahnya
terlalu sedikit untuk diurus dan membuang baris latih malah mengurangi informasi.

Sekarang bagian yang paling menentukan: berapa besar sumbangan tiap rentang harga terhadap
MAE. Kami pakai tebakan konstan median sebagai patokan kasar.

In [ ]:
bins = [0, 1e5, 3e5, 6e5, 1e6, 2e6, 5e6, 1e9]
labels = ["<=100rb", "100-300rb", "300-600rb", "600rb-1jt", "1-2jt", "2-5jt", ">5jt"]
grp = pd.cut(y, bins, labels=labels)

err_const = np.abs(y - np.median(y))
tab = pd.DataFrame({"kelompok": grp, "err": err_const, "y": y}).groupby("kelompok", observed=True).agg(
    n=("y", "size"), mae=("err", "mean"), total=("err", "sum"))
tab["pangsa_mae"] = tab.total / tab.total.sum()
tab["pangsa_baris"] = tab.n / tab.n.sum()

fig, ax = plt.subplots(figsize=(9, 4))
xp = np.arange(len(tab))
ax.bar(xp - 0.2, tab.pangsa_baris*100, width=0.38, color=BLUE, label="porsi jumlah listing")
ax.bar(xp + 0.2, tab.pangsa_mae*100, width=0.38, color=ORANGE, label="porsi sumbangan MAE")
for i, (a, b) in enumerate(zip(tab.pangsa_baris*100, tab.pangsa_mae*100)):
    ax.text(i-0.2, a+0.8, f"{a:.0f}%", ha="center", fontsize=8, color=INK2)
    ax.text(i+0.2, b+0.8, f"{b:.0f}%", ha="center", fontsize=8, color=INK2)
ax.set_xticks(xp); ax.set_xticklabels(tab.index, rotation=20, ha="right")
ax.legend(frameon=False, loc="upper right")
tidy(ax, "Listing mahal jumlahnya sedikit tapi menguasai MAE", None, "persen")
plt.tight_layout(); plt.show()

print(tab.assign(mae=tab.mae.map("{:,.0f}".format),
                 pangsa_mae=(tab.pangsa_mae*100).map("{:.1f}%".format),
                 pangsa_baris=(tab.pangsa_baris*100).map("{:.1f}%".format))[["n","mae","pangsa_baris","pangsa_mae"]])

Inilah masalah sebenarnya. Listing di atas 2 juta cuma 5,8 persen dari data tapi menyumbang
sekitar setengah dari total MAE. Yang di atas 5 juta jumlahnya 217 baris, 1,5 persen, dan
menyumbang sekitar sepertiga.

Artinya perbaikan 10 persen pada listing kelas menengah nyaris tidak menggerakkan skor,
sementara perbaikan di segmen mahal langsung terasa. Kami pegang ini sebagai arah kerja.

## 3. Apa yang bisa diambil dari teksnya

Sekarang ke fiturnya. Pertanyaan pertama yang jelas: teks yang panjang itu tandanya apa.

In [ ]:
tlen = train.text.str.len().values
print(f"panjang teks: min {tlen.min()}, median {np.median(tlen):.0f}, max {tlen.max()}")
print(f"korelasi log(panjang) dengan log(harga): {np.corrcoef(np.log(tlen), ly)[0,1]:.3f}")

kat = pd.qcut(tlen, 5)
kuintil = pd.qcut(tlen, 5, labels=[f"Q{i+1}" for i in range(5)])
med = pd.DataFrame({"k": kuintil, "y": y}).groupby("k", observed=True).y.median()
tepi = list(zip(kat.categories.left, kat.categories.right))

fig, ax = plt.subplots(figsize=(9, 4))
shades = ["#cfe0f6", "#a8c6ee", "#7aa8e3", "#4a8bda", "#2a78d6"]
b = ax.bar(range(5), med.values, color=shades, edgecolor="white", linewidth=1.5)
for i, v in enumerate(med.values):
    ax.text(i, v + 12000, f"{v:,.0f}", ha="center", fontsize=9, color=INK)
ax.set_xticks(range(5))
ax.set_xticklabels([f"{lab}\n{int(a)}-{int(b)} huruf" for lab, (a, b) in zip(med.index, tepi)])
ax.yaxis.set_major_formatter(FMT)
tidy(ax, "Makin panjang deskripsinya, makin mahal propertinya", None, "median harga")
plt.tight_layout(); plt.show()

Naik terus dari 350 ribu ke 779 ribu. Masuk akal kalau dipikir lagi, agen menulis panjang
lebar kalau memang ada yang bisa dijual: kolam renang, wine cellar, pemandangan. Rumah
biasa cukup tiga kalimat.

Korelasinya 0,395 di ruang log, dan itu lebih kuat dari kebanyakan fitur angka yang berhasil
kami ekstrak. Fitur yang kelihatan sepele ternyata salah satu yang paling berguna.

### Ekstraksi angka dengan regex

Berikutnya kami coba menarik angka terstruktur dari teks: jumlah kamar, kamar mandi, luas,
luas lahan, tahun bangun, garasi. Ini bagian yang paling banyak makan waktu karena penulisan
agen tidak seragam. Satu orang menulis "3 bedrooms", yang lain "3BR", yang lain lagi
"three bedroom". Pola regexnya kami perlebar sampai semua varian itu tertangkap.

In [ ]:
NUM_WORD = {"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
            "eleven":11,"twelve":12}

def _num(x):
    try: return float(str(x).replace(",", ""))
    except Exception: return np.nan

def _all(pat, s):
    return [v for v in (_num(m) for m in re.findall(pat, s, re.I)) if v == v]

def _one(pat, s):
    m = re.search(pat, s, re.I)
    return _num(m.group(1)) if m else np.nan

def angka_dari_teks(s):
    l = s.lower()
    l2 = re.sub(r"\b(" + "|".join(NUM_WORD) + r")\b", lambda m: str(NUM_WORD[m.group(1)]), l)
    d = {}

    bd = [v for v in _all(r"(\d+)[\s-]*(?:bed\s*rooms?|bedrooms?|beds?\b|br\b|bd\b|bdrm)", l2) if 0 < v < 25]
    d["beds"] = float(max(set(bd), key=bd.count)) if bd else np.nan

    ba = [v for v in _all(r"(\d+\.?\d*)[\s-]*(?:bath\s*rooms?|bathrooms?|baths?\b|ba\b)", l2) if 0 < v < 25]
    d["baths"] = float(max(set(ba), key=ba.count)) if ba else np.nan
    d["full_bath"] = _one(r"(\d+)[\s-]*full[\s-]*bath", l2)
    d["half_bath"] = _one(r"(\d+)[\s-]*half[\s-]*bath", l2)

    sq = _all(r"([\d,]{3,7})\s*(?:\+/-\s*)?(?:sq\.?\s*ft|sqft|sq\.?\s*feet|square\s*f|sf\b)", l)
    sq_ok = [v for v in sq if 100 <= v <= 60000] or sq
    d["sqft"] = float(max(sq_ok)) if sq_ok else np.nan

    ac = _all(r"(\d[\d,]*\.?\d*)\s*(?:\+/-\s*)?acres?\b", l)
    ac_ok = [v for v in ac if 0 < v <= 20000] or ac
    d["acres"] = float(max(ac_ok)) if ac_ok else np.nan

    d["lot_sqft"] = _one(r"lot[^.]{0,30}?([\d,]{4,8})\s*(?:sq|sf)", l)
    d["garage"] = _one(r"(\d)[\s-]*car\s*garage", l2)
    d["stories"] = _one(r"(\d)[\s-]*(?:story|stories|level)", l2)
    d["units"] = _one(r"(\d+)[\s-]*(?:unit|plex)", l2)
    d["fireplace"] = _one(r"(\d+)\s*fireplace", l2)

    tahun = [int(v) for v in re.findall(r"\b(1[6-9]\d\d|20[0-2]\d)\b", l) if 1700 <= int(v) <= 2026]
    m = re.search(r"built (?:in )?(1[6-9]\d\d|20[0-2]\d)", l)
    d["year"] = float(m.group(1)) if m else (float(tahun[0]) if tahun else np.nan)

    dol = _all(r"\$\s?([\d,]{4,12})", s)
    d["dollar_max"] = max(dol) if dol else np.nan
    d["dollar_n"] = len(dol)
    d["hoa"] = _one(r"hoa[^.]{0,20}?\$?\s?([\d,]{2,6})", l)

    d["sqft_per_bed"] = d["sqft"] / d["beds"] if d.get("beds") else np.nan
    d["log_sqft"] = np.log1p(d["sqft"]) if d["sqft"] == d["sqft"] else np.nan
    d["log_acres"] = np.log1p(d["acres"]) if d["acres"] == d["acres"] else np.nan
    return d

contoh = angka_dari_teks(train.text.iloc[1])
print("contoh hasil ekstraksi:")
print({k: v for k, v in contoh.items() if v == v})

In [ ]:
t0 = time.time()
ang_tr = pd.DataFrame([angka_dari_teks(t) for t in train.text])
ang_te = pd.DataFrame([angka_dari_teks(t) for t in test.text])
print(f"selesai dalam {time.time()-t0:.0f} detik")

cak = ang_tr[["beds","baths","sqft","acres","year","garage","stories","fireplace","hoa"]].notna().mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(cak)), cak.values*100, color=BLUE, height=0.65)
for i, v in enumerate(cak.values*100):
    ax.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=9, color=INK)
ax.set_yticks(range(len(cak))); ax.set_yticklabels(cak.index)
ax.set_xlim(0, 100)
tidy(ax, "Berapa persen listing yang menyebut angka ini", "persen listing", None, grid_axis="x")
plt.tight_layout(); plt.show()

Hasilnya jauh dari lengkap. Jumlah kamar cuma ketemu di 52 persen listing, kamar mandi 33
persen, dan luas bangunan cuma 19 persen. Sisanya memang tidak pernah disebut agennya.

Jadi fitur regex saja tidak akan cukup. Lebih dari separuh data tidak punya satupun angka
luas, dan justru di situ model berbasis teks harus bekerja. Ini yang membuat kami tidak
berhenti di tabular dan lanjut ke TF-IDF serta embedding.

Tapi sebelum itu, ada informasi lain yang bisa diambil dari teks tanpa berupa angka.

### Penanda kata

Balik lagi ke temuan bahwa MAE dikuasai dua ekor. Jadi kami bikin penanda biner untuk
kata-kata yang menandai listing murah dan listing mahal. Daftarnya disusun setelah membaca
puluhan listing di kedua ujung harga.

In [ ]:
MURAH = {
    "auction":  r"\bauction|online bidding|bidding (?:opens|ends)",
    "as_is":    r"\bas[- ]is\b|sold as is|no repairs",
    "fixer":    r"\bfixer|handyman|needs? (?:work|tlc|updating|repair)|\btlc\b|rehab",
    "distress": r"foreclosur|short sale|bank[- ]owned|\breo\b|estate sale|probate",
    "lahan":    r"vacant land|raw land|buildable lot|undeveloped|build your dream",
    "mobile":   r"mobile home|manufactured home|park model|doublewide",
    "cash":     r"cash only|investor (?:special|opportunity)|not financeable",
}
MAHAL = {
    "air":      r"waterfront|lakefront|oceanfront|beachfront|riverfront|private (?:dock|beach)",
    "estate":   r"\bestate\b|compound|manor|chateau|villa\b|mansion",
    "lahan_ls": r"\bacreage|sprawling|rolling (?:hills|pasture)|homestead|vineyard|orchard",
    "kuda":     r"equestrian|horse (?:property|barn)|stable|paddock|pasture",
    "mewah":    r"luxur|custom[- ]built|designer|bespoke|high[- ]end|no expense",
    "fasilitas":r"wine cellar|elevator|infinity pool|home theater|chef'?s kitchen|guest house|casita",
    "gerbang":  r"gated|private (?:drive|gate)|guard",
    "view":     r"panoramic|breathtaking|sweeping views?|mountain views?|golf course",
    "penthouse":r"penthouse|top floor|doorman|concierge",
}
TIPE = {
    "condo":    r"\bcondo|condominium",
    "town":     r"town(?:house|home)|row house",
    "multi":    r"duplex|triplex|fourplex|multi[- ]family|income propert",
    "kabin":    r"\bcabin\b|cottage|bungalow|chalet",
    "historis": r"victorian|colonial|craftsman|tudor|farmhouse|historic|circa",
}

def penanda(s):
    l = s.lower()
    d = {}
    for pre, grp in (("m_", MURAH), ("h_", MAHAL), ("t_", TIPE)):
        for k, p in grp.items():
            d[pre + k] = int(bool(re.search(p, l)))
    d["n_murah"] = sum(d["m_" + k] for k in MURAH)
    d["n_mahal"] = sum(d["h_" + k] for k in MAHAL)
    return d

def gaya(s):
    kata = s.split()
    return {"panjang": len(s), "n_kata": len(kata),
            "n_kalimat": s.count(".") + s.count("!") + s.count("?"),
            "n_redact": s.count("[Redacted"), "n_seru": s.count("!"), "n_tanya": s.count("?"),
            "n_digit": sum(c.isdigit() for c in s),
            "rasio_kapital": sum(c.isupper() for c in s) / max(1, len(s)),
            "n_kata_kapital": len(re.findall(r"\b[A-Z]{4,}\b", s)),
            "rata_kata": np.mean([len(w) for w in kata]) if kata else 0.0,
            "n_propn": len(re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]{2,}\b", s))}

pen_tr = pd.DataFrame([penanda(t) for t in train.text])
pen_te = pd.DataFrame([penanda(t) for t in test.text])
gay_tr = pd.DataFrame([gaya(t) for t in train.text])
gay_te = pd.DataFrame([gaya(t) for t in test.text])

FIT_TR = pd.concat([ang_tr, pen_tr, gay_tr], axis=1).astype(float)
FIT_TE = pd.concat([ang_te, pen_te, gay_te], axis=1).astype(float).reindex(columns=FIT_TR.columns, fill_value=0.0)
print("jumlah fitur tabular:", FIT_TR.shape[1])

In [ ]:
rows = []
for c in pen_tr.columns:
    if c.startswith(("m_", "h_", "t_")) and pen_tr[c].sum() >= 100:
        mk = pen_tr[c] > 0
        rows.append((c, int(mk.sum()), np.median(y[mk]), np.corrcoef(pen_tr[c], ly)[0, 1]))
rows.sort(key=lambda r: r[3])
lab = [r[0] for r in rows]; cor = [r[3] for r in rows]; med_ = [r[2] for r in rows]

fig, ax = plt.subplots(figsize=(9, 6))
warna = [ORANGE if c < 0 else BLUE for c in cor]
ax.barh(range(len(rows)), cor, color=warna, height=0.68)
for i, (c, m) in enumerate(zip(cor, med_)):
    off = 0.006 if c >= 0 else -0.006
    ax.text(c + off, i, f"med {m/1000:,.0f} rb", va="center",
            ha="left" if c >= 0 else "right", fontsize=8, color=INK2)
ax.axvline(0, color=INK2, linewidth=1)
ax.set_yticks(range(len(rows))); ax.set_yticklabels(lab)
ax.set_xlim(min(cor)-0.09, max(cor)+0.09)
tidy(ax, f"Korelasi penanda kata dengan log harga (median keseluruhan {np.median(y)/1000:,.0f} rb)",
     "korelasi", None, grid_axis="x")
plt.tight_layout(); plt.show()

Bekerja persis seperti dugaan. Listing yang menyebut wine cellar atau elevator median
harganya 1,19 juta, lebih dari dua kali median keseluruhan. Yang menyebut vacant land
mediannya 190 ribu. Penanda "mewah" korelasinya 0,304, dan `n_mahal` yang cuma menghitung
berapa banyak penanda mahal yang muncul korelasinya 0,394. Itu lebih kuat dari fitur angka
manapun kecuali jumlah kamar mandi.

Catatan jujur: setelah masuk ke model akhir, tambahan fitur-fitur ini ternyata cuma
memperbaiki CV sekitar 60 poin MAE, praktis nol. Penyebabnya TF-IDF sudah menangkap
kata-kata ini sebagai token biasa. Jadi fitur ini bagus buat dipahami datanya, tapi bukan
sumber kenaikan skor. Kami tetap pakai karena tidak merugikan dan membantu model pohon
membuat interaksi dengan ukuran properti.

## 4. Apakah train dan test datang dari sebaran yang sama

Ini pemeriksaan yang hampir kami lewatkan, dan ternyata jadi temuan paling berguna di
notebook ini.

Caranya dengan adversarial validation: gabung train dan test, beri label 0 dan 1, lalu latih
classifier untuk membedakan keduanya. Kalau AUC-nya 0,5 berarti sebarannya identik dan CV
kita bisa dipercaya apa adanya. Kalau jauh di atas 0,5 berarti ada pergeseran dan CV akan
menipu.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

gab = pd.concat([train.text, test.text], ignore_index=True)
lab_adv = np.r_[np.zeros(len(train)), np.ones(len(test))]

vec_adv = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
X_adv = vec_adv.fit_transform(gab)
p_adv = cross_val_predict(LogisticRegression(max_iter=1000), X_adv, lab_adv, cv=3, method="predict_proba")[:, 1]
auc = roc_auc_score(lab_adv, p_adv)
print(f"AUC train vs test : {auc:.4f}")
print()
pat_red = r"\[Redacted"
red_tr = train.text.str.contains(pat_red).mean()
red_te = test.text.str.contains(pat_red).mean()
len_tr = train.text.str.len().median()
len_te = test.text.str.len().median()
print(f"panjang teks  train {len_tr:.0f}  |  test {len_te:.0f}")
print(f"ada [Redacted] train {red_tr:.3f}  |  test {red_te:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins_ = np.linspace(0, 3000, 60)
ax.hist(train.text.str.len().clip(upper=3000), bins=bins_, density=True, color=BLUE,
        alpha=0.75, label="train", edgecolor="white", linewidth=0.3)
ax.hist(test.text.str.len().clip(upper=3000), bins=bins_, density=True, color=ORANGE,
        alpha=0.6, label="test", edgecolor="white", linewidth=0.3)
ax.axvline(train.text.str.len().median(), color=BLUE, linewidth=2, linestyle="--")
ax.axvline(test.text.str.len().median(), color=ORANGE, linewidth=2, linestyle="--")
ax.legend(frameon=False)
tidy(ax, "Teks di test lebih panjang daripada di train", "panjang teks (huruf)", "kerapatan")
plt.tight_layout(); plt.show()

AUC-nya 0,594. Bukan angka yang bisa diabaikan. Teks di test lebih panjang (median 924 lawan
871 huruf) dan lebih sering mengandung `[Redacted Entity]` (17,8 persen lawan 14,0 persen).

Dan kita sudah tahu dari bagian sebelumnya bahwa teks panjang berarti properti mahal. Jadi
test condong ke arah properti yang lebih mahal daripada train, yang berarti ekornya lebih
berat, yang berarti MAE-nya akan lebih besar.

Ini menjelaskan sesuatu yang bikin kami bingung sebelumnya: CV kami turun terus tapi skor
leaderboard tidak ikut turun sebanding. Ternyata kami mengukur dengan penggaris yang salah.

Solusinya, kami hitung bobot importance sampling dari probabilitas classifier tadi, lalu
pakai bobot itu untuk dua hal: menimbang ulang metrik validasi supaya mendekati sebaran
test, dan menimbang sampel saat melatih stacker.

In [ ]:
bobot = np.clip(p_adv[:len(train)] / np.clip(1 - p_adv[:len(train)], 1e-6, None), 0, 8)
bobot = bobot / bobot.mean()

print(f"bobot: p10 {np.quantile(bobot,0.1):.2f}  median {np.median(bobot):.2f}  p90 {np.quantile(bobot,0.9):.2f}  max {bobot.max():.2f}")
print(f"rata-rata harga biasa          : {y.mean():,.0f}")
print(f"rata-rata harga setelah dibobot: {np.average(y, weights=bobot):,.0f}")

def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

def mae_test(a, b):
    return float(np.average(np.abs(np.asarray(a) - np.asarray(b)), weights=bobot))

Setelah dibobot, rata-rata harga naik dari 839 ribu ke 933 ribu. Dugaan kami terkonfirmasi.

Mulai sekarang setiap angka validasi kami laporkan dua kali: CV biasa, dan CV ala-test yang
sudah ditimbang. Yang kedua itu yang kami pakai untuk mengambil keputusan.

## 5. Strategi validasi

Beberapa keputusan yang kami ambil di sini, semuanya karena bentuk datanya:

Foldnya StratifiedKFold 5 lipat, distratifikasi pada 20 kuantil log harga. Kalau pakai
KFold biasa, satu fold bisa kebagian listing 80 juta lebih banyak dari yang lain dan
skornya jadi berayun tidak karuan. Stratifikasi membuat tiap fold punya komposisi ekor yang
mirip.

Satu set fold dipakai semua model, dari TF-IDF sampai stacker. Ini syarat supaya stacking
tidak bocor.

Semua model dilatih pada log harga dengan loss L1 atau Huber, sesuai alasan di bagian 1.

Stacker punya masalah tambahan: kalau early stopping-nya memakai fold validasi yang sama
dengan yang dipakai menilai OOF, skornya jadi optimistis. Jadi early stopping kami ambil
dari potongan 10 persen di dalam data latih tiap fold, bukan dari fold validasinya.

In [ ]:
from sklearn.model_selection import StratifiedKFold

strata = pd.qcut(ly, q=20, labels=False, duplicates="drop")
fold = np.zeros(len(train), dtype=int)
for k, (_, v) in enumerate(StratifiedKFold(NFOLD, shuffle=True, random_state=SEED).split(ly, strata)):
    fold[v] = k

cek = pd.DataFrame({"fold": fold, "y": y}).groupby("fold").agg(
    n=("y", "size"), median=("y", "median"), mean=("y", "mean"), p99=("y", lambda s: s.quantile(0.99)))
print(cek.round(0).astype("int64").to_string())

print()
print("tebakan konstan median, sebagai lantai pembanding:")
print(f"  MAE biasa    : {mae(y, np.full(len(y), np.median(y))):,.0f}")
print(f"  MAE ala-test : {mae_test(y, np.full(len(y), np.median(y))):,.0f}")

## 6. Model

Rancangannya bertingkat dua.

Tingkat satu berisi banyak model sederhana yang masing-masing melihat teks dari sudut
berbeda: TF-IDF kata, TF-IDF karakter, tetangga terdekat, dan embedding dari model bahasa.
Tiap model menghasilkan prediksi out-of-fold.

Tingkat dua adalah LightGBM yang memakai prediksi-prediksi itu bersama fitur tabular tadi
sebagai input. Model pohon di sini gunanya membuat interaksi, misalnya menimbang prediksi
tetangga terdekat lebih berat kalau kemiripannya tinggi.

Kenapa tidak langsung satu model besar saja: sudah kami coba, dan stacking menang cukup
jauh. Angkanya ada di bagian evaluasi.

### Tingkat 1a: TF-IDF dan tetangga terdekat

Enam model di sini. Yang menarik adalah blok kNN: karena beberapa listing di dataset ini
hampir kembar (rumah yang sama diiklankan ulang dengan sedikit perubahan kata), mencari
tetangga terdekat berdasarkan kemiripan kosinus TF-IDF kadang langsung memberi harga yang
sangat dekat. Kami tidak cuma ambil prediksinya, tapi juga skor kemiripannya, supaya
stacker bisa belajar kapan harus mempercayainya.

In [ ]:
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVR

def norm_angka(t):
    return re.sub(r"\d", "0", t.lower())

def propn(t):
    return " ".join(re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]{2,}\b", t))

def blok_knn(Q, R, yr, k=10):
    out = []
    for s in range(0, Q.shape[0], 1000):
        S = (Q[s:s+1000] @ R.T).toarray()
        kk = min(k, S.shape[1] - 1)
        idx = np.argpartition(-S, kk, axis=1)[:, :kk]
        sim = np.take_along_axis(S, idx, 1)
        o = np.argsort(-sim, axis=1)
        idx, sim = np.take_along_axis(idx, o, 1), np.take_along_axis(sim, o, 1)
        p = yr[idx]
        w = np.maximum(sim, 1e-6) ** 4
        out.append(np.c_[(p*w).sum(1)/w.sum(1), np.median(p, 1), p[:, 0],
                         sim[:, 0], sim[:, :3].mean(1), p[:, :3].mean(1), p.std(1)])
    return np.vstack(out)

KOLOM = (["tfidf_cw", "tfidf_w", "knn_bobot", "knn_median", "knn_1", "knn_sim1", "knn_sim3",
          "knn_mean3", "knn_std"] + [f"svd{i}" for i in range(32)] + ["svr_linear", "ridge_propn", "ridge_norm"])

def tingkat1_teks(teks_latih, y_latih, teks_target):
    vw = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True)
    vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=200000, sublinear_tf=True)
    Wa, Wb = vw.fit_transform(teks_latih), vw.transform(teks_target)
    Ca, Cb = vc.fit_transform(teks_latih), vc.transform(teks_target)

    p_cw = Ridge(alpha=3.0).fit(hstack([Wa, Ca]).tocsr(), y_latih).predict(hstack([Wb, Cb]).tocsr())
    p_w = Ridge(alpha=1.0).fit(Wa, y_latih).predict(Wb)
    kn = blok_knn(Wb, Wa, y_latih)
    sv = TruncatedSVD(32, random_state=0).fit(Wa).transform(Wb)

    vn = TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_features=300000, sublinear_tf=True)
    Da = vn.fit_transform(pd.Series(teks_latih).map(norm_angka))
    Db = vn.transform(pd.Series(teks_target).map(norm_angka))
    p_svr = LinearSVR(C=0.3, epsilon=0.0, max_iter=5000, random_state=0).fit(Da, y_latih).predict(Db)
    p_rn = Ridge(alpha=2.0).fit(Da, y_latih).predict(Db)

    vp = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    Pa = vp.fit_transform(pd.Series(teks_latih).map(propn))
    p_pr = Ridge(alpha=1.0).fit(Pa, y_latih).predict(vp.transform(pd.Series(teks_target).map(propn)))

    return pd.DataFrame(np.c_[p_cw, p_w, kn, sv, p_svr, p_pr, p_rn], columns=KOLOM)

In [ ]:
t0 = time.time()
oof_teks = pd.DataFrame(np.zeros((len(train), len(KOLOM))), columns=KOLOM)
test_teks = np.zeros((len(test), len(KOLOM)))

for k in range(NFOLD):
    trn, val = np.where(fold != k)[0], np.where(fold == k)[0]
    target = pd.concat([train.text.iloc[val], test.text], ignore_index=True)
    F = tingkat1_teks(train.text.iloc[trn].values, ly[trn], target.values)
    oof_teks.iloc[val] = F.iloc[:len(val)].values
    test_teks += F.iloc[len(val):].values / NFOLD
    print(f"fold {k} selesai, {time.time()-t0:.0f} detik")

print()
for c in ["tfidf_w", "tfidf_cw", "knn_bobot", "svr_linear", "ridge_norm", "ridge_propn"]:
    p = np.exp(oof_teks[c].values)
    print(f"  {c:14s} CV {mae(y, p):>9,.0f}   ala-test {mae_test(y, p):>9,.0f}")

Model teks terbaik sendirian ada di kisaran 350 ribu, turun dari lantai 550 ribu. Belum
bagus, tapi ini baru bahan untuk stacker.

Perhatikan jarak antara kolom CV dan kolom ala-test, sekitar 40 ribu. Itu ongkos pergeseran
distribusi yang tadi ditemukan, dan jaraknya konsisten di semua model.

### Tingkat 1b: embedding kalimat

TF-IDF cuma melihat kata sebagai simbol. Dia tidak tahu "breathtaking mountain views" dan
"stunning alpine vistas" itu maksudnya sama. Untuk itu kami pakai model embedding yang sudah
dilatih pada korpus besar, ambil vektornya tanpa fine-tune, lalu pasang model ringan di
atasnya.

Dua model yang dipakai, `bge-large-en-v1.5` dan `e5-large-v2`. Keduanya kelas large, dan
keduanya dari keluarga arsitektur yang berbeda supaya prediksinya tidak terlalu seragam.

Di atas embedding kami pasang tiga kepala: RidgeCV, kNN kosinus, dan SVR dengan kernel RBF.
SVR yang paling kuat tapi paling lambat, jadi lima foldnya kami jalankan paralel pakai
joblib. Tanpa itu tahap ini makan setengah jam lebih per model.

In [ ]:
import torch
import torch.nn.functional as Fn
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from joblib import Parallel, delayed

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)
if DEV == "cuda":
    print(torch.cuda.get_device_name(0))

def embed(nama, teks, pool="cls", prefix="", max_len=512, bs=32):
    tok = AutoTokenizer.from_pretrained(nama)
    mdl = AutoModel.from_pretrained(nama)
    if DEV == "cuda":
        mdl = mdl.half()
    mdl = mdl.to(DEV).eval()

    enc = tok([prefix + t for t in teks], truncation=True, max_length=max_len)["input_ids"]
    urut = np.argsort([len(e) for e in enc])
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    out = np.zeros((len(teks), mdl.config.hidden_size), dtype=np.float32)

    t0 = time.time()
    with torch.no_grad():
        for bi, s in enumerate(range(0, len(urut), bs)):
            ids = urut[s:s+bs]
            L = max(len(enc[i]) for i in ids)
            x = torch.full((len(ids), L), pad, dtype=torch.long)
            m = torch.zeros((len(ids), L), dtype=torch.long)
            for j, i in enumerate(ids):
                x[j, :len(enc[i])] = torch.tensor(enc[i])
                m[j, :len(enc[i])] = 1
            h = mdl(input_ids=x.to(DEV), attention_mask=m.to(DEV)).last_hidden_state.float()
            if pool == "cls":
                v = h[:, 0]
            else:
                mm = m.to(DEV).unsqueeze(-1).float()
                v = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
            out[ids] = Fn.normalize(v, dim=-1).cpu().numpy()
            if bi and bi % 150 == 0:
                sel = s + bs
                print(f"    {sel}/{len(urut)}  sisa sekitar {(len(urut)-sel)/(sel/(time.time()-t0))/60:.1f} menit")

    del mdl
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return out

def svr_satu_fold(k, Etr, Ete, ly, fold):
    trn, val = np.where(fold != k)[0], np.where(fold == k)[0]
    mu, sd = ly[trn].mean(), ly[trn].std()
    m = SVR(C=3.0, epsilon=0.05).fit(Etr[trn], (ly[trn] - mu) / sd)
    return k, val, m.predict(Etr[val]) * sd + mu, m.predict(Ete) * sd + mu

def kepala_embedding(E, tag):
    n = len(train)
    Etr, Ete = E[:n], E[n:]
    oof = {h: np.zeros(n) for h in ["ridge", "knn", "svr"]}
    tst = {h: np.zeros(len(test)) for h in ["ridge", "knn", "svr"]}

    for k in range(NFOLD):
        trn, val = np.where(fold != k)[0], np.where(fold == k)[0]
        r = RidgeCV(alphas=np.logspace(-2, 1, 7)).fit(Etr[trn], ly[trn])
        oof["ridge"][val] = r.predict(Etr[val])
        tst["ridge"] += r.predict(Ete) / NFOLD
        for Q, dst, idx in ((Etr[val], oof, val), (Ete, tst, None)):
            S = Q @ Etr[trn].T
            top = np.argpartition(-S, 20, axis=1)[:, :20]
            sim = np.take_along_axis(S, top, 1)
            w = np.exp((sim - sim.max(1, keepdims=True)) / 0.02)
            p = (w * ly[trn][top]).sum(1) / w.sum(1)
            if idx is not None:
                dst["knn"][idx] = p
            else:
                dst["knn"] += p / NFOLD

    hasil = Parallel(n_jobs=min(NFOLD, os.cpu_count()), backend="loky")(
        delayed(svr_satu_fold)(k, Etr, Ete, ly, fold) for k in range(NFOLD))
    for k, val, pv, pt in hasil:
        oof["svr"][val] = pv
        tst["svr"] += pt / NFOLD

    for h in oof:
        print(f"  {tag}_{h:6s} CV {mae(y, np.exp(oof[h])):>9,.0f}   ala-test {mae_test(y, np.exp(oof[h])):>9,.0f}")
    return ({f"{tag}_{h}": oof[h] for h in oof}, {f"{tag}_{h}": tst[h] for h in tst})

In [ ]:
MODEL_EMB = [
    ("bge_l", "BAAI/bge-large-en-v1.5", "cls", ""),
    ("e5_l", "intfloat/e5-large-v2", "mean", "query: "),
]

semua_teks = list(train.text) + list(test.text)
oof_emb, test_emb, emb_cache = {}, {}, {}

for tag, nama, pool, prefix in MODEL_EMB:
    t0 = time.time()
    print(f"[{tag}] {nama}")
    try:
        E = embed(nama, semua_teks, pool=pool, prefix=prefix)
        emb_cache[tag] = E
        print(f"  embedding selesai {time.time()-t0:.0f} detik, dimensi {E.shape[1]}")
        o, t = kepala_embedding(E, tag)
        oof_emb.update(o)
        test_emb.update(t)
    except Exception as e:
        print(f"  GAGAL: {type(e).__name__}: {str(e)[:200]}")
    print(f"  total {time.time()-t0:.0f} detik\n")

SVR di atas embedding jadi model tunggal terbaik kami. Ridge di atas embedding yang sama
justru paling lemah, yang menarik karena artinya hubungan antara vektor embedding dan harga
memang tidak linear.

Satu tambahan yang murah: komponen SVD dari embedding mentah kami masukkan juga sebagai
fitur stacker, bukan cuma prediksi kepalanya. Model pohon jadi bisa mengiris ruang embedding
sendiri dan mengaitkannya dengan fitur ukuran properti.

In [ ]:
komponen = {}
for tag, E in emb_cache.items():
    sv = TruncatedSVD(48, random_state=0).fit(E[:len(train)])
    A, B = sv.transform(E[:len(train)]), sv.transform(E[len(train):])
    for j in range(A.shape[1]):
        komponen[f"emb_{tag}_svd{j}"] = (A[:, j], B[:, j])
    print(f"{tag}: 48 komponen, varian tertangkap {sv.explained_variance_ratio_.sum():.3f}")

del emb_cache
gc.collect()

### Tingkat 2: stacking

Semua prediksi OOF dan fitur tabular digabung jadi satu matriks, lalu dua LightGBM dilatih
di atasnya, satu dengan objective L1 dan satu dengan Huber. Hasil keduanya dirata-rata.

Dua hal yang perlu disebut soal cara melatihnya. Pertama, early stopping diambil dari
potongan dalam data latih, bukan dari fold validasi, supaya skor OOF-nya jujur. Kedua,
sample weight-nya pakai bobot adversarial dari bagian 4, jadi stacker lebih memperhatikan
baris yang mirip test.

Model finalnya dilatih ulang di seluruh data latih dengan tiga seed berbeda lalu dirata-rata,
untuk meredam variasi acak LightGBM.

In [ ]:
import lightgbm as lgb

fitur = pd.concat([
    FIT_TR.reset_index(drop=True),
    oof_teks.reset_index(drop=True),
    pd.DataFrame({k: v for k, v in oof_emb.items()}),
    pd.DataFrame({k: v[0] for k, v in komponen.items()}),
], axis=1)

fitur_test = pd.concat([
    FIT_TE.reset_index(drop=True),
    pd.DataFrame(test_teks, columns=KOLOM),
    pd.DataFrame({k: v for k, v in test_emb.items()}),
    pd.DataFrame({k: v[1] for k, v in komponen.items()}),
], axis=1).reindex(columns=fitur.columns, fill_value=0.0)

print("matriks stacker:", fitur.shape)

PARAM = dict(learning_rate=0.03, num_leaves=31, min_data_in_leaf=20, feature_fraction=0.7,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=-1)
OBJ = {"l1": dict(PARAM, objective="l1"), "huber": dict(PARAM, objective="huber", alpha=1.0)}

oof_stack, n_iter = {}, {}
for nama, par in OBJ.items():
    o, its = np.zeros(len(fitur)), []
    for k in range(NFOLD):
        trn, val = np.where(fold != k)[0], np.where(fold == k)[0]
        rng = np.random.default_rng(k)
        dalam = rng.choice(trn, size=len(trn)//10, replace=False)
        inti = np.setdiff1d(trn, dalam)
        m = lgb.train(par, lgb.Dataset(fitur.iloc[inti], ly[inti], weight=bobot[inti]), 3000,
                      valid_sets=[lgb.Dataset(fitur.iloc[dalam], ly[dalam], weight=bobot[dalam])],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        o[val] = m.predict(fitur.iloc[val], num_iteration=m.best_iteration)
        its.append(m.best_iteration)
    oof_stack[nama] = o
    n_iter[nama] = max(50, int(np.mean(its) * 1.1))
    print(f"  {nama:6s} CV {mae(y, np.exp(o)):>9,.0f}   ala-test {mae_test(y, np.exp(o)):>9,.0f}   iter {n_iter[nama]}")

oof_final = (oof_stack["l1"] + oof_stack["huber"]) / 2
print()
print(f"  blend  CV {mae(y, np.exp(oof_final)):>9,.0f}   ala-test {mae_test(y, np.exp(oof_final)):>9,.0f}")

In [ ]:
pred_log = np.zeros(len(test))
model_akhir = []
for nama, par in OBJ.items():
    for s in range(3):
        m = lgb.train(dict(par, seed=s, bagging_seed=s, feature_fraction_seed=s),
                      lgb.Dataset(fitur, ly, weight=bobot), n_iter[nama])
        pred_log += m.predict(fitur_test) / (3 * len(OBJ))
        model_akhir.append(m)
pred = np.exp(np.clip(pred_log, 0, np.log(3e8)))
print("prediksi test siap:", pred.shape)

## 7. Evaluasi

Pertama, perbandingan semua model yang dipakai supaya kelihatan sumbangan tiap tingkat.

In [ ]:
oof_semua = {}
for c in ["tfidf_w", "knn_bobot", "svr_linear"]:
    oof_semua[c] = oof_teks[c].values
oof_semua.update(oof_emb)
oof_semua["STACK (final)"] = oof_final

skor = pd.DataFrame([(k, mae(y, np.exp(v)), mae_test(y, np.exp(v))) for k, v in oof_semua.items()],
                    columns=["model", "cv", "cv_ala_test"]).sort_values("cv_ala_test", ascending=False)

fig, ax = plt.subplots(figsize=(9, max(4, 0.42*len(skor))))
warna = [AQUA if "STACK" in m else BLUE for m in skor.model]
ax.barh(range(len(skor)), skor.cv_ala_test, color=warna, height=0.68)
for i, (v, m) in enumerate(zip(skor.cv_ala_test, skor.model)):
    ax.text(v + 4000, i, f"{v:,.0f}", va="center", fontsize=9,
            color=INK, fontweight="bold" if "STACK" in m else "normal")
ax.axvline(mae_test(y, np.full(len(y), np.median(y))), color=ORANGE, linestyle="--", linewidth=2)
ax.text(mae_test(y, np.full(len(y), np.median(y))), len(skor)-0.3, " tebakan konstan",
        color=ORANGE, fontsize=9, va="top")
ax.set_yticks(range(len(skor))); ax.set_yticklabels(skor.model)
ax.set_xlim(0, skor.cv_ala_test.max()*1.22)
ax.xaxis.set_major_formatter(FMT)
tidy(ax, "MAE ala-test tiap model (makin kecil makin baik)", "MAE", None, grid_axis="x")
plt.tight_layout(); plt.show()

print(skor.assign(cv=skor.cv.map("{:,.0f}".format),
                  cv_ala_test=skor.cv_ala_test.map("{:,.0f}".format)).to_string(index=False))

Stacking menang telak dari model tunggal terbaiknya. Jarak itu yang membenarkan arsitektur
dua tingkat: masing-masing model salah di tempat yang berbeda, dan LightGBM belajar kapan
harus percaya pada siapa.

In [ ]:
pred_oof = np.exp(oof_final)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

ax = axes[0]
idx = np.random.default_rng(0).choice(len(y), min(3000, len(y)), replace=False)
ax.scatter(y[idx], pred_oof[idx], s=7, alpha=0.28, color=BLUE, linewidths=0)
lim = [2e4, 9e7]
ax.plot(lim, lim, color=ORANGE, linewidth=2, linestyle="--")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
tidy(ax, "Prediksi OOF vs harga sebenarnya", "harga sebenarnya", "prediksi", grid_axis="both")

ax = axes[1]
des = pd.qcut(pred_oof, 10, labels=False)
kal = pd.DataFrame({"d": des, "p": pred_oof, "y": y}).groupby("d").median()
rasio = (kal.y / kal.p).values
ax.bar(range(10), rasio, color=[AQUA if 0.9 <= r <= 1.1 else ORANGE for r in rasio], width=0.7)
ax.axhline(1.0, color=INK2, linewidth=1.5)
for i, r in enumerate(rasio):
    ax.text(i, r + 0.015, f"{r:.2f}", ha="center", fontsize=8, color=INK)
ax.set_ylim(0, 1.35); ax.set_xticks(range(10))
ax.set_xticklabels([f"D{i+1}" for i in range(10)])
tidy(ax, "Kalibrasi: median aktual dibagi prediksi, per desil prediksi", "desil prediksi", "rasio")

plt.tight_layout(); plt.show()

Panel kanan itu pemeriksaan yang menurut kami paling penting dan paling sering dilewatkan.

Kalau kita kelompokkan berdasarkan harga sebenarnya, model selalu terlihat menyusut ke
tengah: properti mahal diprediksi terlalu murah, properti murah terlalu mahal. Dulu kami
kira itu bug dan sempat mau menaikkan prediksi di segmen mahal.

Ternyata itu artefak statistik. Mengelompokkan berdasarkan hasil selalu memunculkan pola
seperti itu, bahkan pada model yang sempurna sekalipun. Arah yang benar adalah
mengelompokkan berdasarkan prediksi, dan di panel kanan rasionya duduk di sekitar 1,0.
Modelnya sudah terkalibrasi.

Untuk memastikan, kami uji langsung apakah menaikkan prediksi di ujung atas membantu.

In [ ]:
atas = pred_oof >= np.quantile(pred_oof, 0.9)
print("pengali pada desil prediksi teratas:")
print(f"  {'pengali':>8}  {'CV':>10}  {'ala-test':>10}")
for m in [0.9, 1.0, 1.05, 1.1, 1.25, 1.5]:
    p = pred_oof.copy()
    p[atas] = p[atas] * m
    tag = "  <- sekarang" if m == 1.0 else ""
    print(f"  {m:>8.2f}  {mae(y, p):>10,.0f}  {mae_test(y, p):>10,.0f}{tag}")

Terbukti. Naik 10 persen cuma menggeser MAE beberapa ratus poin, dan begitu dinaikkan 25
persen ke atas skornya langsung memburuk. Ide itu kami buang.

Ini alasan kenapa model L1 di ruang log memang jawaban yang benar untuk MAE. Menyusut ke
tengah saat model tidak yakin bukan kelemahan, itu justru perilaku optimal.

In [ ]:
pot = [0, 1e5, 3e5, 6e5, 1e6, 2e6, 5e6, 1e9]
nama_pot = ["<=100rb", "100-300rb", "300-600rb", "600rb-1jt", "1-2jt", "2-5jt", ">5jt"]
kel = pd.cut(y, pot, labels=nama_pot)
err = np.abs(y - pred_oof)

d = pd.DataFrame({"kelompok": kel, "err": err, "y": y, "p": pred_oof})
g = d.groupby("kelompok", observed=True)
ringkas = pd.DataFrame({
    "n": g.size(),
    "mae": g.err.mean(),
    "pangsa_mae": g.err.sum() / err.sum() * 100,
    "rasio_pred_aktual": g.p.median() / g.y.median(),
})

print("Analisis error berdasarkan harga sebenarnya:")
tampil = ringkas.copy()
tampil["mae"] = tampil.mae.map("{:,.0f}".format)
tampil["pangsa_mae"] = tampil.pangsa_mae.map("{:.1f}%".format)
tampil["rasio_pred_aktual"] = tampil.rasio_pred_aktual.map("{:.2f}".format)
print(tampil.to_string())
print()
print(f"MAE untuk y <= 2 juta : {err[y <= 2e6].mean():,.0f}")
print(f"MAE untuk y >  2 juta : {err[y > 2e6].mean():,.0f}")
print(f"median error persentase absolut: {np.median(np.abs(y - pred_oof)/y)*100:.1f}%")

In [ ]:
imp = np.zeros(fitur.shape[1])
for m in model_akhir:
    imp += m.feature_importance("gain")
imp = pd.Series(imp / len(model_akhir), index=fitur.columns).sort_values(ascending=False).head(20)[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(imp)), imp.values, color=BLUE, height=0.7)
ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f"{v/1e3:.0f}k"))
tidy(ax, "20 fitur paling berpengaruh di stacker (gain)", "total gain", None, grid_axis="x")
plt.tight_layout(); plt.show()

Yang di atas hampir semuanya prediksi dari tingkat 1, terutama SVR di atas embedding dan
blok kNN. Fitur tabular buatan sendiri ada di daftar tapi tidak mendominasi.

Itu sekali lagi memperkuat kesimpulan tadi: untuk data ini, representasi teks yang bagus
jauh lebih berharga daripada rekayasa fitur manual.

## 8. Submission

In [ ]:
sub = sample[["id"]].merge(pd.DataFrame({"id": test.id.values, "listPrice": np.round(pred, 2)}),
                           on="id", how="left")

assert len(sub) == len(sample), "jumlah baris tidak cocok"
assert sub.listPrice.notna().all(), "ada NaN"
assert (sub.listPrice > 0).all(), "ada nilai nol atau negatif"
assert (sub.id.values == sample.id.values).all(), "urutan id tidak cocok"

sub.to_csv("submission.csv", index=False)
print("submission.csv tersimpan\n")
print(sub.head(8).to_string(index=False))
print()
print(f"baris        : {len(sub)}")
print(f"minimum      : {sub.listPrice.min():,.0f}")
print(f"median       : {sub.listPrice.median():,.0f}   (train: {np.median(y):,.0f})")
print(f"rata-rata    : {sub.listPrice.mean():,.0f}   (train: {y.mean():,.0f})")
print(f"maksimum     : {sub.listPrice.max():,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
b = np.linspace(4, 8, 60)
ax.hist(np.log10(np.clip(y, 1, None)), bins=b, density=True, color=BLUE, alpha=0.75,
        label="harga train", edgecolor="white", linewidth=0.3)
ax.hist(np.log10(sub.listPrice), bins=b, density=True, color=ORANGE, alpha=0.6,
        label="prediksi test", edgecolor="white", linewidth=0.3)
ax.legend(frameon=False)
ax.set_xticks(range(4, 9)); ax.set_xticklabels([f"$10^{i}$" for i in range(4, 9)])
tidy(ax, "Sebaran prediksi dibanding sebaran harga latih", "harga", "kerapatan")
plt.tight_layout(); plt.show()

Sebaran prediksinya lebih rapat daripada sebaran aslinya, dan itu memang seharusnya begitu.
Prediksi adalah median bersyarat, jadi variansnya pasti lebih kecil dari varians target.
Kalau sebaran prediksi kami sama persis dengan sebaran harga latih, justru itu tanda model
kami memaksakan diri menebak ekor dan akan dihukum oleh MAE.

## 9. Catatan penutup

### Yang berhasil

Stacking dua tingkat memberi lompatan terbesar. Model tunggal terbaik ada di kisaran 340an
ribu, stacker menembus jauh di bawahnya.

Embedding kalimat menyumbang kenaikan paling besar di antara semua jenis model tingkat satu.
Dari TF-IDF saja ke TF-IDF ditambah embedding, CV kami turun sekitar 12 ribu poin. Itu
lebih besar dari total semua perbaikan lain digabung.

Memeriksa pergeseran train dan test ternyata bukan formalitas. Tanpa itu kami akan terus
mengoptimalkan angka yang salah dan bingung kenapa leaderboard tidak bergerak.

### Yang dicoba tapi tidak berhasil

Kami catat yang gagal juga, karena hasilnya menghemat waktu orang lain.

Menimbang sampel dengan harga, supaya model lebih memperhatikan properti mahal, malah
memperburuk CV dari 309 ribu ke 318 ribu dengan bobot akar harga dan 348 ribu dengan bobot
harga penuh. Secara teori memang begitu seharusnya, karena L1 tanpa bobot sudah menghasilkan
median bersyarat yang optimal untuk MAE.

Target encoding nama tempat yang diambil dari proper noun di teks. Di atas model sederhana
kelihatan menjanjikan, memberi sekitar 3400 poin. Setelah dipasang di stacker penuh,
sisanya tinggal 500 poin. TF-IDF sudah menangkap nama kota sebagai token biasa.

Mengoreksi prediksi dengan pengali di segmen mahal, sudah ditunjukkan di bagian 7, hasilnya
nol sampai negatif.

Menyalin harga dari listing kembar untuk baris yang punya tetangga sangat mirip. Terdengar
masuk akal karena memang ada listing yang hampir identik, tapi gainnya cuma sekitar 600
sampai 1400 poin dan stacker sudah punya fitur kNN yang menangkap sebagian besarnya.

Fine-tune ModernBERT penuh. Sebagai model tunggal hasilnya 347 ribu, kalah dari SVR di atas
embedding beku yang 346 ribu, padahal waktu latihnya lebih dari satu jam dibanding setengah
jam. Sumbangannya ke stacker cuma sekitar 1900 poin. Untuk anggaran waktu lomba, embedding
beku jauh lebih efisien.

### Kalau waktunya lebih panjang

Yang paling ingin kami coba adalah melatih transformer sampai benar-benar konvergen. Saat
kami hentikan di epoch ketiga, MAE validasinya masih turun 7 sampai 19 ribu tiap epoch, yang
artinya modelnya belum selesai belajar. Kemungkinan besar masih ada ruang di situ.

Selain itu, model embedding yang lebih besar lagi. Pola di data ini konsisten menunjukkan
kualitas representasi teks adalah pengungkit utama, bukan rekayasa fitur.